In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import subprocess
from pathlib import Path
from IPython.display import Image, display
import warnings
import sys
warnings.filterwarnings('ignore')

from mimiciii_db import DB
from mimiciii_db.config import db_url


In [23]:
db = DB.from_url(db_url())
print("Database connected successfully!")


Database connected successfully!


In [24]:
# Load subgroup assignments
subgroup_df = pd.read_csv("../../../../../data/lca_k6_patient_level_with_class.csv")
print(f"Subgroups shape: {subgroup_df.shape}")
print(f"Subgroup distribution:")
print(subgroup_df['latent_class'].value_counts().sort_index())
subgroup_df.head()


Subgroups shape: (36606, 39)
Subgroup distribution:
latent_class
1     3455
2    10072
3     2536
4     9448
5     7235
6     3860
Name: count, dtype: int64


,subject_id,hadm_id,gender,age,age_bin,admission_elective_flag,congestive_heart_failure,cardiac_arrhythmias,valvular_disease,pulmonary_circulation,...,fluid_electrolyte,blood_loss_anemia,deficiency_anemias,alcohol_abuse,drug_abuse,psychoses,depression,row_id,age_grp_cat,latent_class
0,58526,100001,F,35.48,25-44,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,25-44,1
1,54610,100003,M,59.91,45-64,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,45-64,6
2,9895,100006,F,48.92,45-64,0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,3,45-64,4
3,23018,100007,F,73.82,65-84,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,65-84,2
4,533,100009,M,60.80,45-64,0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,45-64,2


In [25]:
# Query morbidity counts
query = "SELECT * FROM filtered_patients_with_morbidity_counts"
morbidity_df = db.query_df(query)
print(f"Morbidity counts shape: {morbidity_df.shape}")
print(f"\nMorbidity count distribution:")
print(morbidity_df['morbidity_count'].describe())
morbidity_df.head()


Morbidity counts shape: (36606, 12)

Morbidity count distribution:
count    36429.000000
mean         3.081117
std          1.988860
min          0.000000
25%          2.000000
50%          3.000000
75%          4.000000
max         13.000000
Name: morbidity_count, dtype: float64


,subject_id,hadm_id,icustay_id,gender,icu_intime,icu_outtime,admittime,dischtime,deathtime,admission_type,age,morbidity_count
0,58526,100001,275225,F,2117-09-11 11:47:35,2117-09-15 17:57:14,2117-09-11 11:46:00,2117-09-17 16:45:00,NaT,EMERGENCY,35.48,3.0
1,54610,100003,209281,M,2150-04-17 15:35:42,2150-04-19 14:12:52,2150-04-17 15:34:00,2150-04-21 17:30:00,NaT,EMERGENCY,59.91,2.0
2,9895,100006,291788,F,2108-04-06 15:50:15,2108-04-11 15:18:03,2108-04-06 15:49:00,2108-04-18 17:18:00,NaT,EMERGENCY,48.92,4.0
3,23018,100007,217937,F,2145-03-31 10:17:23,2145-04-04 12:41:10,2145-03-31 05:33:00,2145-04-07 12:40:00,NaT,EMERGENCY,73.82,1.0
4,533,100009,253656,M,2162-05-17 10:18:31,2162-05-19 22:05:14,2162-05-16 15:56:00,2162-05-21 13:37:00,NaT,EMERGENCY,60.80,5.0


In [26]:
# Merge to see what the visualization will use
merged = subgroup_df[['hadm_id', 'latent_class']].merge(
    morbidity_df[['hadm_id', 'morbidity_count']], 
    on='hadm_id', 
    how='inner'
)
print(f"Merged data shape: {merged.shape}")
merged.groupby('latent_class')['morbidity_count'].describe()


Merged data shape: (36606, 3)


,count,mean,std,min,25%,50%,75%,max
latent_class,,,,,,,,
1,3455.0,5.157164,1.642979,1.0,4.0,5.0,6.0,11.0
2,10048.0,2.267715,1.212713,0.0,1.0,2.0,3.0,8.0
3,2536.0,5.924685,1.748329,2.0,5.0,6.0,7.0,12.0
4,9443.0,3.270571,1.495375,0.0,2.0,3.0,4.0,11.0
5,7087.0,1.244814,1.100397,0.0,0.0,1.0,2.0,6.0
6,3860.0,4.380052,1.590124,1.0,3.0,4.0,5.0,13.0


In [27]:
# Generate bubble plot
subprocess.run([sys.executable, "fig_5_bubble.py"], check=True)
print("Bubble plot generated successfully!")


✓ Visualization saved to /Users/guoxuanxu/Documents/local_repo/25fa-dsc180a-team1/assets/fig_5/subgroup_network_group_1.png
✓ Visualization saved to /Users/guoxuanxu/Documents/local_repo/25fa-dsc180a-team1/assets/fig_5/subgroup_network_group_3.png
✓ Visualization saved to /Users/guoxuanxu/Documents/local_repo/25fa-dsc180a-team1/assets/fig_5/subgroup_network_group_4.png
✓ Visualization saved to /Users/guoxuanxu/Documents/local_repo/25fa-dsc180a-team1/assets/fig_5/subgroup_network_group_6.png

✓ All subgroup network visualizations completed!
Bubble plot generated successfully!
